# 00.05 · Researching like a pro: docs, llms.txt, changelogs, model cards

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 75–90 min |
| 📋 Prerequisites | Lessons 00.01–00.04. Internet access (no API key needed). |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Rank** sources of truth: provider model lists and deprecation pages, official guides, API reference, changelogs, GitHub issues, tutorials.
2. **Navigate** docs.langchain.com through its `llms.txt` index and Markdown page versions, and know what the docs MCP servers are for.
3. **Detect** retired model IDs in official examples automatically, by checking pages against a provider's deprecation list.
4. **Read** a changelog, a versioning policy, and a deprecation lifecycle, and decide what they mean for your code.
5. **Read** a model card and extract what matters for an engineering decision.

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [LangChain docs `llms.txt`](https://docs.langchain.com/llms.txt) | Machine-readable index of every docs page |
| [Use docs programmatically (docs MCP)](https://docs.langchain.com/use-these-docs) | The `docs-langchain` and `reference-langchain` MCP servers |
| [LangChain changelog](https://docs.langchain.com/oss/python/releases/changelog) · [Versioning](https://docs.langchain.com/oss/python/versioning) · [Release policy](https://docs.langchain.com/oss/python/release-policy) | What changed, and how stable each API is |
| [Groq deprecations](https://console.groq.com/docs/deprecations) · [Groq models](https://console.groq.com/docs/models) · [Groq `llms.txt`](https://console.groq.com/llms.txt) | The provider's source of truth for model IDs |
| [Gemini release notes](https://ai.google.dev/gemini-api/docs/changelog) · [Gemini deprecations](https://ai.google.dev/gemini-api/docs/deprecations) | Same idea, for Google |
| [Hugging Face model cards](https://huggingface.co/docs/hub/model-cards) · example: [openai/gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b) · [Groq model page: GPT OSS 20B](https://console.groq.com/docs/model/openai/gpt-oss-20b) | How to read a model card |

## Setup

This lesson reads **public** web pages, so it needs internet access but **no API key**. If you are offline, every network cell prints a clear message instead of crashing, and you can still read the explanations.

In [ ]:
# Step 1: Find the repository root.
# WHY: Jupyter starts this kernel in the notebook's own folder (modules/00-foundations/),
# but .env, data/ and src/ live at the repo root, two levels up.
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the course repo (pyproject.toml). "
        "Start Jupyter from inside your clone of langchain-langgraph-mastery."
    )

# Step 2: Make the course package `mastery` importable.
# After `python -m pip install -e .` (Lesson 00.02 guide) the import just works.
# If you skipped that step, we fall back to src/ so the lesson still runs.
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ `mastery` is not installed in this env; loaded it from src/.")
    print("   Fix once (Git Bash, course env active, repo root): python -m pip install -e .")

print("Repo root :", REPO_ROOT)
print("mastery   :", mastery.__version__)

In [ ]:
import json
import re
from importlib import metadata

import httpx  # Installed with `mastery` (pip install -e .). A modern HTTP client.

TIMEOUT_S = 20.0
# Some sites reject requests without a browser-like User-Agent, so we identify ourselves politely.
HEADERS = {"User-Agent": "lc-mastery-course/0.1 (learning notebook)"}


def fetch_text(url: str) -> str | None:
    """Download a public page as text. Return None (and explain) on any failure.

    Args:
        url: Public URL. No authentication is sent.
    """
    try:
        response = httpx.get(url, headers=HEADERS, timeout=TIMEOUT_S, follow_redirects=True)
        response.raise_for_status()
    except httpx.HTTPError as exc:
        print(f"⚠️ Could not fetch {url} ({type(exc).__name__}). Offline, or the page moved?")
        return None
    # WHY this check: some doc sites answer "200 OK" with a "Page Not Found" body (a "soft 404").
    if "404 - Page Not Found" in response.text[:2000]:
        print(f"⚠️ {url} returned a soft 404 (status 200, but the page does not exist).")
        return None
    return response.text


print("httpx", metadata.version("httpx"), "ready.")

---
## 1. "Trust, then verify": ranking your sources

Everything about LLM tooling changes monthly, so you need a habit, not a memory. When two sources disagree, **the one higher in this table wins**:

| Rank | Source | Answers | Example |
|---|---|---|---|
| 1 | **The provider's live API / model list** | "Does this model ID exist *for my account* today?" | `GET /openai/v1/models` (Lesson 00.07) |
| 2 | **Provider deprecation pages and release notes** | "When does it stop working? What replaces it?" | [Groq deprecations](https://console.groq.com/docs/deprecations) |
| 3 | **Official guides** (docs.langchain.com) | "How is this meant to be used?" | [Models](https://docs.langchain.com/oss/python/langchain/models) |
| 4 | **API reference** (reference.langchain.com) | "Exact parameters and types" | [`init_chat_model`](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) |
| 5 | **Changelogs, versioning and release policies** | "What changed? Is it stable?" | [Changelog](https://docs.langchain.com/oss/python/releases/changelog) |
| 6 | **GitHub issues and discussions** | "Is this a known bug? Is there a workaround?" | [langchain issues](https://github.com/langchain-ai/langchain/issues) |
| 7 | Blog posts, videos, courses (including this one), AI assistants | Ideas and context, **always verify** | — |

Notice rank 3 is below rank 1–2: **official guides go stale too.** You will prove that in Section 4.

---
## 2. Navigating docs.langchain.com like a machine: `llms.txt`

`llms.txt` is a plain-text index designed for AI tools, and it is just as useful for people: one file that lists every section of the docs. LangChain's version is hierarchical: the top file links to smaller indexes (Build, Deploy, Monitor…), which link to pages.

In [ ]:
LLMS_TXT = "https://docs.langchain.com/llms.txt"

index = fetch_text(LLMS_TXT)
if index:
    lines = index.splitlines()
    print(f"{LLMS_TXT}: {len(lines)} lines\n")
    # Index entries look like: - [TITLE (N pages)](https://docs.langchain.com/_llms/....txt): description
    found = re.findall(r"^- \[([^\]]+)\]\((https://docs\.langchain\.com/_llms/[^)]+)\)", index, re.M)
    # dict.fromkeys() removes duplicates but keeps the original order (the file lists some twice).
    for title_, url in list(dict.fromkeys(found))[:8]:
        print(f"  {title_:<62} {url}")

**You should now see** a short file (a few dozen lines) with indexes such as *AGENT DEVELOPMENT LIFECYCLE / Build / Python*. Let's follow that one and search it.

In [ ]:
PYTHON_BUILD_INDEX = "https://docs.langchain.com/_llms/agent-development-lifecycle/build/python.txt"


def search_index(index_url: str, keyword: str, limit: int = 8) -> list[str]:
    """Return up to `limit` lines of a docs index that mention `keyword` (case-insensitive)."""
    text = fetch_text(index_url)
    if text is None:
        return []
    return [line for line in text.splitlines() if keyword.lower() in line.lower()][:limit]


for line in search_index(PYTHON_BUILD_INDEX, "groq"):
    print(line[:160])

### Markdown versions of pages

Every page on docs.langchain.com is also served as **Markdown** if you add `.md` to its URL (the docs' own MCP setup prompt uses `https://docs.langchain.com/use-these-docs.md`). Markdown is easier to search, diff, and paste into notes than HTML:

In [ ]:
page = fetch_text("https://docs.langchain.com/oss/python/langchain/install.md")
if page:
    # Print only the lines that state a requirement, instead of the whole page.
    wanted = [line.strip() for line in page.splitlines() if "Requires Python" in line or line.startswith("# ")]
    for line in dict.fromkeys(wanted):   # The page repeats the requirement for pip and uv tabs.
        print(line)

### The docs MCP servers

LangChain runs two public **MCP servers** (Model Context Protocol, Module 05) so that coding assistants can query the docs live instead of relying on stale training data ([Use docs programmatically](https://docs.langchain.com/use-these-docs)):

| Server | URL | Covers |
|---|---|---|
| `docs-langchain` | `https://docs.langchain.com/mcp` | Guides, how-tos, tutorials for LangChain, LangGraph, LangSmith |
| `reference-langchain` | `https://reference.langchain.com/mcp` | API reference: classes, methods, parameters |

If you use an AI coding assistant (Cursor, VS Code, Claude Code, and others), connect **both** following that page. You don't need them for the course, but they make "check the current docs" a one-question task.

---
## 3. The provider is the source of truth: Groq's model list

Groq's model page is also available as Markdown. Each model links to `/docs/model/<id>`, which makes the IDs easy to extract. We split the page into its **Production** and **Preview** sections.

In [ ]:
MODEL_LINK = re.compile(r"\]\(/docs/model/([^)\s]+)\)(Enterprise)?")


def groq_models_from_docs() -> dict[str, list[str]]:
    """Parse console.groq.com/docs/models.md into {'production': [...], 'preview': [...]}.

    Models marked 'Enterprise' are only for enterprise contracts, so we label them.
    """
    text = fetch_text("https://console.groq.com/docs/models.md")
    if text is None:
        return {}
    sections = {"production": text.split("## [Production Models]")[-1].split("## [Preview Models]")[0],
                "preview": text.split("## [Preview Models]")[-1].split("## [Deprecated Models]")[0]}
    return {name: [m_id + (" (Enterprise only)" if ent else "") for m_id, ent in MODEL_LINK.findall(body)]
            for name, body in sections.items()}


docs_models = groq_models_from_docs()
for section, ids in docs_models.items():
    print(f"{section.upper()} ({len(ids)}):")
    for model_id in ids:
        print("   ", model_id)

### What just happened?

- **Production** models are "intended for use in your production environments"; **preview** models are "intended for evaluation purposes only" and "may be discontinued at short notice" (both quotes from the [Groq models page](https://console.groq.com/docs/models)).
- Some production entries are marked **Enterprise only**: for example `llama-3.3-70b-versatile` still exists for enterprise contracts, but its deprecation "applies to free and developer-tier usage" ([deprecations](https://console.groq.com/docs/deprecations)). So the same ID can work for one company and fail for you. That's why rank 1 is *your account's* live list (Lesson 00.07).
- Parsing a docs page is a research shortcut. The page layout can change any day; the Models API is the stable interface.

---
## 4. Case study: official examples that use retired model IDs

Below is a **hand-curated** list from the [Groq deprecations page](https://console.groq.com/docs/deprecations). Keeping such a list (with dates and sources) is a normal engineering practice: it lets you scan code and docs automatically.

In [ ]:
# VERIFIED 2026-10-11: https://console.groq.com/docs/deprecations
# Shutdown dates apply to free and developer tiers (enterprise contracts may differ).
GROQ_RETIRED: dict[str, tuple[str, str]] = {   # id -> (shutdown date, recommended replacement)
    "groq/compound": ("2026-09-21", "—"),
    "groq/compound-mini": ("2026-09-21", "—"),
    "qwen/qwen3.6-27b": ("2026-09-14", "qwen/qwen3.8-27b"),
    "llama-3.1-8b-instant": ("2026-08-16", "openai/gpt-oss-20b"),
    "llama-3.3-70b-versatile": ("2026-08-16", "openai/gpt-oss-120b"),
    "qwen/qwen3-32b": ("2026-07-17", "openai/gpt-oss-120b"),
    "meta-llama/llama-4-scout-17b-16e-instruct": ("2026-07-17", "openai/gpt-oss-120b"),
    "moonshotai/kimi-k2-instruct-0905": ("2026-04-15", "openai/gpt-oss-120b"),
    "meta-llama/llama-4-maverick-17b-128e-instruct": ("2026-03-09", "openai/gpt-oss-120b"),
    "meta-llama/llama-guard-4-12b": ("2026-03-05", "openai/gpt-oss-safeguard-20b"),
    "playai-tts": ("2025-12-31", "canopylabs/orpheus-v1-english"),
    "playai-tts-arabic": ("2025-12-31", "canopylabs/orpheus-arabic-saudi"),
    "moonshotai/kimi-k2-instruct": ("2025-10-10", "openai/gpt-oss-120b"),
    "gemma2-9b-it": ("2025-10-08", "llama-3.1-8b-instant (itself retired since)"),
}
print(f"{len(GROQ_RETIRED)} retired Groq model IDs in the curated list.")

In [ ]:
def find_ids(text: str, ids: list[str]) -> list[str]:
    """Return the IDs that appear in `text` as whole tokens.

    WHY not a plain `in` check: 'groq/compound' is a substring of 'groq/compound-mini',
    and 'playai-tts' of 'playai-tts-arabic'. The lookarounds require that the
    characters around a match are not part of a longer ID.
    """
    found = []
    for model_id in ids:
        pattern = rf"(?<![\w/.-]){re.escape(model_id)}(?![\w.-])"
        if re.search(pattern, text):
            found.append(model_id)
    return found


# A quick self-test before trusting the function on real pages.
assert find_ids("use groq/compound-mini here", ["groq/compound", "groq/compound-mini"]) == ["groq/compound-mini"]
print("find_ids self-test passed.")

In [ ]:
PAGES_TO_AUDIT = {
    "LangChain · ChatGroq integration": "https://docs.langchain.com/oss/python/integrations/chat/groq.md",
    "Groq · LangChain page": "https://console.groq.com/docs/langchain.md",
    "Groq · Quickstart": "https://console.groq.com/docs/quickstart.md",
    "Groq · API reference": "https://console.groq.com/docs/api-reference.md",
}

audit: dict[str, list[str]] = {}
for label, url in PAGES_TO_AUDIT.items():
    text = fetch_text(url)
    if text is not None:
        audit[label] = find_ids(text, list(GROQ_RETIRED))

for label, retired_ids in audit.items():
    verdict = "✅ none found" if not retired_ids else "❌ uses " + ", ".join(
        f"{i} (shut down {GROQ_RETIRED[i][0]})" for i in retired_ids)
    print(f"{label:<34} {verdict}")

### What just happened?

On **2026-10-11** this audit found retired IDs in **every one** of these official pages: the LangChain ChatGroq page used `qwen/qwen3-32b` (in its main example) plus the two retired Llama 4 vision models; Groq's own LangChain page and quickstart used `llama-3.3-70b-versatile`; Groq's API reference used `llama-3.1-8b-instant`, `llama-3.3-70b-versatile`, `playai-tts`, and `gemma2-9b-it` in its examples. Your result may differ if the pages were fixed since: **that is the point of running it live**.

Lessons:

1. **Official ≠ current.** Example code is the part of the docs updated least often.
2. **Copy patterns, not model IDs.** Take the *code shape* from the docs, and the *model ID* from the provider's live list (`.env` → `mastery.config`).
3. **Automate the check.** A 30-line script (this section) catches what eyes miss. Project 00 turns it into a reusable explorer; Module 12 runs checks like this in CI.

### The same thing happens with Google

The LangChain [Models](https://docs.langchain.com/oss/python/langchain/models) page shows `google_genai:gemini-3.7-flash`. Google's [release notes](https://ai.google.dev/gemini-api/docs/changelog) for **October 8, 2026** say `gemini-3.7-flash` "is deprecated and has been replaced by `gemini-3.8-flash`", and that requests to it "are automatically routed" to the new model. Code keeps working, **but silently runs a different model** than the one you tested.

> **Read the column headers.** On Google's [deprecations page](https://ai.google.dev/gemini-api/docs/deprecations) the date next to `gemini-3.7-flash` is its **release** date (August 13, 2026), not its deprecation date. The deprecation is announced in the release notes. Misreading a table is the most common research mistake, including in courses.

In [ ]:
page = fetch_text("https://docs.langchain.com/oss/python/langchain/models.md")
if page:
    gemini_ids = sorted(set(re.findall(r"gemini-[0-9][\w.-]*", page)))
    print("Gemini model IDs mentioned on the LangChain Models page:", gemini_ids)

---
## 5. Versions: PyPI is the source of truth for packages

The PyPI JSON API (`https://pypi.org/pypi/<package>/json`) returns the latest release of any package. Compare it with what is installed and with the course minimums in [`requirements.txt`](../../requirements.txt).

In [ ]:
REQUIREMENTS = (REPO_ROOT / "requirements.txt").read_text(encoding="utf-8")
PACKAGES = ["langchain", "langchain-core", "langgraph", "langchain-groq", "langsmith", "python-dotenv"]


def course_minimum(package: str) -> str:
    """Return the minimum version pinned in requirements.txt (e.g. '1.4.4'), or '?'."""
    match = re.search(rf"^{re.escape(package)}>=([\w.]+)", REQUIREMENTS, re.M)
    return match.group(1) if match else "?"


print(f"{'package':<16} {'course min':<11} {'installed':<14} latest on PyPI")
for package in PACKAGES:
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = "not installed"
    data = fetch_text(f"https://pypi.org/pypi/{package}/json")
    latest = json.loads(data)["info"]["version"] if data else "?"
    print(f"{package:<16} {course_minimum(package):<11} {installed:<14} {latest}")

**You should now see** LangChain packages as "not installed" (they arrive in Lesson 00.07) and the latest versions on PyPI. If *latest* is newer than *course min*, that's normal: minimums are a floor. If the **major** version (first number) is higher, read the changelog before upgrading (next section).

---
## 6. Changelogs, versioning, and release policy

The LangChain [Versioning](https://docs.langchain.com/oss/python/versioning) page says packages follow **semantic versioning** `MAJOR.MINOR.PATCH`: major = breaking changes, minor = backward-compatible features, patch = fixes. It also labels APIs:

| Label | Meaning (from the Versioning page) | What you do |
|---|---|---|
| Stable (no prefix) | Backward compatible; breaking changes only in major releases | Use freely |
| **Beta** | Feature-complete, "may undergo minor changes" | Use, pin versions, watch the changelog |
| **Alpha** | Experimental | Prototypes only |
| **Deprecated** | Will be removed in a future major release | Migrate now (follow the migration guide) |

The previous major version gets security and critical fixes for **12 months** after the next major release. Now let's read the latest changelog entries programmatically:

In [ ]:
changelog = fetch_text("https://docs.langchain.com/oss/python/releases/changelog.md")
if changelog:
    # Entries look like: <Update label="Sep 1, 2026" tags={["langchain"]} ...>  followed by "## `langchain` v1.4.0"
    entries = re.findall(r'<Update label="([^"]+)" tags=\{\[([^\]]*)\]\}.*?\n\s*## (.+?)\n', changelog, re.S)
    for date_label, tags, heading in entries[:6]:
        print(f"{date_label:<14} {tags.replace(chr(34), ''):<26} {heading.strip()}")

### What just happened?

The top entry (at the time of writing: **Sep 1, 2026, `langchain` v1.4.0**) announces that MCP support now ships inside LangChain in the `langchain.mcp` namespace, replacing the standalone `langchain-mcp-adapters` package, and that the namespace is **beta**. One changelog line changes which package Module 05 installs. That is why every module ends with a research checkpoint.

---
## 7. The deprecation lifecycle

Groq documents a four-phase process ([deprecations](https://console.groq.com/docs/deprecations)):

1. **Announcement**: email to affected users plus a docs update with a recommended replacement.
2. **Transition**: the model still works; test the replacement now.
3. **Automatic upgrade** (sometimes): requests may be routed to the replacement.
4. **End-of-life**: requests to the old ID return errors.

Google's Gemini API does the same with "deprecated" → "shut down", and announces deprecations in its release notes. Your code should survive every phase: model IDs in config (not code), a tested fallback model, alerts on "model not found" errors, and a scheduled check against the live model list (Project 00).

---
## 8. Reading a model card

A **model card** documents a model: what it is for, how it was evaluated, its limits, and its license ([Hugging Face model cards](https://huggingface.co/docs/hub/model-cards)). For engineering decisions, read it with these questions, using [GPT OSS 20B on Groq](https://console.groq.com/docs/model/openai/gpt-oss-20b) and [openai/gpt-oss-20b on Hugging Face](https://huggingface.co/openai/gpt-oss-20b) as examples:

| Question | Where to look | GPT OSS 20B on Groq (2026-10-11) |
|---|---|---|
| What goes in and out? | Modalities | Text in, text out |
| Can it call tools / return JSON? | Capabilities | Tool use, browser search, code execution, JSON object and JSON schema modes |
| How much context? | Limits | 131,072-token context; up to 65,536 output tokens ([models](https://console.groq.com/docs/models)) |
| What does it cost? | Pricing | $0.075 input / $0.30 output per 1M tokens |
| Production or preview? | Provider list | **Production** |
| License, intended use, known limits | The publisher's card (Hugging Face) | Read the license and the limitations section yourself |

> **Provider page ≠ publisher card.** Groq's page tells you how *Groq* serves the model (speed, limits, price). The publisher's card (OpenAI's, on Hugging Face) tells you how the model was trained and evaluated, and its license. You need both.

### GitHub issues: the source for "is it just me?"

When an error message makes no sense, search it in the project's issues, for example `https://github.com/langchain-ai/langchain/issues?q=<your error text>`. Prefer issues with a maintainer reply or a linked fix, check the **version** they mention, and treat workarounds as temporary.

---
## ⚠️ Pitfalls

| Trap | Better habit |
|---|---|
| Copying a model ID from a tutorial (or an official example) | Take IDs from the provider's live list; keep them in `.env` |
| Trusting a date without reading the table header | Check what the column means (release vs deprecation vs shutdown) |
| Reading only the guide, not the changelog | Skim the changelog for every package you upgrade |
| Upgrading a major version "because it's newer" | Read the migration guide first; pin versions in `requirements.txt` |
| Believing an AI assistant's answer about an API | Ask it to cite the docs page, then open that page (or connect the docs MCP servers) |

## ⚠️ Production notes

> ⚠️ **Production note: scraping docs pages is a research shortcut.**
> This notebook parses HTML/Markdown pages whose layout can change without notice. In production: use the provider's **API** (Groq Models API, PyPI JSON API), cache results, and run the check on a schedule (CI job or cron) that alerts a human, rather than parsing web pages at request time. See Project 00 and Module 12.

## 🏋️ Exercises

1. **(Easy)** Use `search_index` to find the docs page about `init_chat_model`. Open it in your browser.
2. **(Easy)** Add the [Groq text generation page](https://console.groq.com/docs/text-chat) (`.md` version) to `PAGES_TO_AUDIT`. Does it use retired IDs?
3. **(Medium)** Write `audit_repo(paths)` that scans this repository's `.md` and `.ipynb` files for IDs in `GROQ_RETIRED` and lists any hits. (Hint: `REPO_ROOT.rglob("*.md")`. Expect hits only where a page *explains* a retirement.)
4. **(Stretch)** Turn Section 6 into `latest_changes(package)` that returns only changelog entries tagged with that package (e.g. `"langgraph"`).

<details>
<summary><b>Show solutions</b></summary>

**3.**
```python
def audit_repo(patterns: tuple[str, ...] = ("*.md", "*.ipynb")) -> dict[str, list[str]]:
    hits = {}
    for pattern in patterns:
        for path in REPO_ROOT.rglob(pattern):
            if "_maintainer" in path.parts or ".ipynb_checkpoints" in path.parts:
                continue
            found = find_ids(path.read_text(encoding="utf-8", errors="ignore"), list(GROQ_RETIRED))
            if found:
                hits[str(path.relative_to(REPO_ROOT))] = found
    return hits

for file, ids in audit_repo().items():
    print(file, ids)
```

**4.**
```python
def latest_changes(package: str, limit: int = 5) -> list[tuple[str, str]]:
    return [(d, h.strip()) for d, tags, h in entries if f'"{package}"' in tags][:limit]
print(latest_changes("langgraph"))
```
</details>

## ✅ Checkpoint

- [ ] You can name the top three sources of truth for a model ID, in order.
- [ ] You ran the audit in Section 4 and know what it found today.
- [ ] You can explain *beta* vs *deprecated* in LangChain's versioning policy.
- [ ] You know the difference between a provider's model page and a publisher's model card.

## 🔎 Research checkpoint

Open [Groq deprecations](https://console.groq.com/docs/deprecations). Find the most recent deprecation and its recommended replacement. Is the replacement a production or a preview model ([models](https://console.groq.com/docs/models))? Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Provider live lists and deprecation pages beat official examples, which beat tutorials.
- `llms.txt`, `.md` page versions, and the docs MCP servers make the LangChain docs searchable by you and your tools.
- Read changelogs and versioning labels before upgrading; read column headers before quoting dates.
- A small script that checks docs against a curated deprecation list is real engineering. Project 00 builds on it.

**Next:** [00.06 · Free vs paid vs local: a decision framework](00-06-provider-decision-framework.ipynb).